In [30]:
# import os
# from langchain_community.document_loaders import DirectoryLoader, TextLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.vectorstores import FAISS
# from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
# from langchain_core.prompts import PromptTemplate
# from langchain_classic.chains import create_retrieval_chain
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
# import torch

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import faiss
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from pathlib import Path


import numpy as np

In [21]:
def load_lean_files(directory_path):
    print(f"Recursively scanning '{directory_path}' for Lean files...")
    docs = []
    # rglob handles deeply nested directories automatically
    for path in Path(directory_path).rglob("*.lean"):
        try:
            # Explicit utf-8 prevents crashes on math unicode symbols like ∀, ∃, ⊢
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
                docs.append(Document(page_content=text, metadata={"source": str(path)}))
        except Exception as e:
            print(f"Skipping {path} due to error: {e}")
    print(f"Successfully loaded {len(docs)} Lean documents.")
    return docs

docs = load_lean_files("../mathlib4/Mathlib")

Recursively scanning '../mathlib4/Mathlib' for Lean files...
Successfully loaded 8125 Lean documents.


In [22]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        r"\nnamespace\s+",
        r"\nsection\s+",
        r"\ntheorem\s+",
        r"\nlemma\s+",
        r"\ndef\s+",
        r"\ninductive\s+",
        r"\nstructure\s+",
        r"\nexample\s+",
        r"\nopen\s+",      # Sometimes splitting at open statements is clean
        r"\n\n", 
        r"\n", 
        r" "
    ],
    is_separator_regex=True,
    chunk_size=1000,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(docs)

In [23]:
chunks[1052]

Document(metadata={'source': '../mathlib4/Mathlib/Probability/StrongLaw.lean'}, page_content="theorem strong_law_Lp {p : ℝ≥0∞} (hp : 1 ≤ p) (hp' : p ≠ ∞) (X : ℕ → Ω → E)\n    (hℒp : MemLp (X 0) p μ) (hindep : Pairwise ((· ⟂ᵢ[μ] ·) on X))\n    (hident : ∀ i, IdentDistrib (X i) (X 0) μ μ) :\n    Tendsto (fun (n : ℕ) => eLpNorm (fun ω => (n : ℝ)⁻¹ • (∑ i ∈ range n, X i ω) - μ[X 0]) p μ)\n      atTop (𝓝 0) := by\n  -- First exclude the trivial case where the space is not a probability space\n  by_cases h : ∀ᵐ ω ∂μ, X 0 ω = 0\n  · have I : ∀ᵐ ω ∂μ, ∀ i, X i ω = 0 := by\n      rw [ae_all_iff]\n      intro i\n      exact (hident i).symm.ae_snd (p := fun x ↦ x = 0) measurableSet_eq h\n    have A (n : ℕ) : eLpNorm (fun ω => (n : ℝ)⁻¹ • (∑ i ∈ range n, X i ω) - μ[X 0]) p μ = 0 := by\n      simp only [integral_eq_zero_of_ae h, sub_zero]\n      apply eLpNorm_eq_zero_of_ae_zero\n      filter_upwards [I] with ω hω\n      simp [hω]\n    simp [A]\n  -- Then use ae convergence and uniform integrability

In [27]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8474.01it/s]


In [28]:
chunk_texts = [chunk.page_content for chunk in chunks]
raw_embeddings = embeddings.embed_documents(chunk_texts)

In [31]:
embedding_matrix = np.array(raw_embeddings).astype('float32')
dimension = embedding_matrix.shape[1]

In [32]:
embedding_matrix

array([[-0.06723551, -0.04746328,  0.02147776, ...,  0.0136079 ,
         0.05253498, -0.09785162],
       [-0.04612501, -0.06744232, -0.00997163, ..., -0.09303449,
         0.0186366 ,  0.07938801],
       [-0.06612404, -0.0594927 ,  0.00579799, ..., -0.10589179,
         0.01730193,  0.08371744],
       ...,
       [-0.08850229,  0.036189  , -0.048315  , ...,  0.0985111 ,
        -0.03020992,  0.02201156],
       [ 0.01226701,  0.05500139,  0.00396227, ...,  0.05582722,
        -0.01845515,  0.0291146 ],
       [-0.04347345,  0.03891918,  0.00911088, ...,  0.07861403,
         0.00101739,  0.01707399]], shape=(141874, 384), dtype=float32)

In [33]:
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embedding_matrix)